In [2]:
from rdflib import Graph, Namespace, URIRef, Literal

# Step 1: Load ChEBI Ontology (Local)
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"  # Update this if needed
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")  # Parse OWL file

# Step 2: Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Step 3: Define Relevant Filters
MOLECULAR_ENTITY = URIRef("http://purl.obolibrary.org/obo/CHEBI_23367")  # Only process molecular entities
RELEVANT_RELATIONS = {
    URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"): "is a subclass of",
    URIRef("http://purl.obolibrary.org/obo/RO_0000087"): "has role",  # 'has role'
    URIRef("http://purl.obolibrary.org/obo/BFO_0000050"): "is part of"  # 'part of'
}

# Step 4: Extract Labels for Only Relevant Entities
chebi_labels = {}
for s, p, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#label"), None)):
    chebi_id = str(s).split("/")[-1]  # Extract ID
    chebi_labels[chebi_id] = str(o)  # Store label

# Step 5: Restrict Processing to Relevant Entities
relevant_entities = set()
for s, p, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"), MOLECULAR_ENTITY)):
    relevant_entities.add(s)  # Keep only molecular entities

# Step 6: Extract and Filter Inference Rules
filtered_facts = set()
for s, p, o in ontology_graph.triples((None, None, None)):
    if s in relevant_entities and p in RELEVANT_RELATIONS:
        filtered_facts.add((s, RELEVANT_RELATIONS[p], o))  # Store filtered facts

# Step 7: Convert to Natural Language
natural_language_statements = []
for s, relation, o in filtered_facts:
    subject_label = chebi_labels.get(str(s).split("/")[-1], str(s))  # Use label if available
    object_label = chebi_labels.get(str(o).split("/")[-1], str(o))  # Use label if available
    natural_language_statements.append(f"{subject_label} {relation} {object_label}.")

# Step 8: Save Results
with open("chebi_filtered_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print(f"Filtered ChEBI facts saved to 'chebi_filtered_facts.txt' with {len(natural_language_statements)} lines.")


Filtered ChEBI facts saved to 'chebi_filtered_facts.txt' with 13 lines.


In [3]:
from rdflib import Graph, Namespace, URIRef, Literal

# Load ChEBI Ontology (Local)
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")

# Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Define Relevant Relationships
MOLECULAR_ENTITY = URIRef("http://purl.obolibrary.org/obo/CHEBI_23367")
RELEVANT_RELATIONS = {
    URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"): "is a subclass of",
    URIRef("http://purl.obolibrary.org/obo/RO_0000087"): "has role",
    URIRef("http://purl.obolibrary.org/obo/BFO_0000050"): "is part of"
}

# Extract Labels Efficiently
chebi_labels = {}
for s, _, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#label"), None)):
    chebi_id = str(s).split("/")[-1]
    chebi_labels[chebi_id] = str(o)

# Filter Relevant Entities (Only Molecular Entities)
relevant_entities = set()
for s, _, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"), MOLECULAR_ENTITY)):
    relevant_entities.add(s)

# Extract & Filter Facts (Skip Blank Nodes)
filtered_facts = set()
for s, p, o in ontology_graph.triples((None, None, None)):
    if s in relevant_entities and p in RELEVANT_RELATIONS:
        if isinstance(o, URIRef) and str(o).startswith("http://purl.obolibrary.org/obo/CHEBI_"):  # Ignore blank nodes
            filtered_facts.add((s, RELEVANT_RELATIONS[p], o))

# Convert to Natural Language (No Duplicates, No Blank Nodes)
natural_language_statements = list(set([
    f"{chebi_labels.get(str(s).split('/')[-1], str(s))} {relation} {chebi_labels.get(str(o).split('/')[-1], str(o))}."
    for s, relation, o in filtered_facts
]))

# Save Results
with open("chebi_cleaned_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print(f"Final cleaned ChEBI facts saved to 'chebi_cleaned_facts.txt' with {len(natural_language_statements)} meaningful facts.")


Final cleaned ChEBI facts saved to 'chebi_cleaned_facts.txt' with 10 meaningful facts.


In [5]:
from rdflib import Graph, Namespace, URIRef, Literal

# Step 1: Load ChEBI Ontology
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")

# Step 2: Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Step 3: Define More Relationships to Extract
RELEVANT_RELATIONS = {
    URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"): "is a subclass of",
    URIRef("http://purl.obolibrary.org/obo/RO_0000087"): "has role",  # 'has role'
    URIRef("http://purl.obolibrary.org/obo/BFO_0000050"): "is part of",  # 'part of'
    URIRef("http://purl.obolibrary.org/obo/CHEBI_50906"): "has functional parent",  # 'has functional parent'
    URIRef("http://purl.obolibrary.org/obo/CHEBI_33232"): "has application",  # 'has application'
    URIRef("http://purl.obolibrary.org/obo/CHEBI_50906"): "has biological role",  # 'has biological role'
}

# Step 4: Extract Labels Efficiently
chebi_labels = {}
for s, _, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#label"), None)):
    chebi_id = str(s).split("/")[-1]
    chebi_labels[chebi_id] = str(o)

# Step 5: Extract Only Relevant Facts (More Than Just Subclass)
filtered_facts = set()
for s, p, o in ontology_graph.triples((None, None, None)):
    if p in RELEVANT_RELATIONS:
        if isinstance(o, URIRef) and str(o).startswith("http://purl.obolibrary.org/obo/CHEBI_"):  # Ignore blank nodes
            filtered_facts.add((s, RELEVANT_RELATIONS[p], o))

# Step 6: Convert to Natural Language
natural_language_statements = list(set([
    f"{chebi_labels.get(str(s).split('/')[-1], str(s))} {relation} {chebi_labels.get(str(o).split('/')[-1], str(o))}."
    for s, relation, o in filtered_facts
]))

# Step 7: Save Results
with open("chebi_expanded_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print(f"Final expanded ChEBI facts saved to 'chebi_expanded_facts.txt' with {len(natural_language_statements)} meaningful facts.")


Final expanded ChEBI facts saved to 'chebi_expanded_facts.txt' with 280873 meaningful facts.


In [6]:
from rdflib import Graph, Namespace, URIRef, Literal

# Load ChEBI Ontology (Local)
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")

# Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Define Relevant Relationships
RELEVANT_RELATIONS = {
    URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"): "is a subclass of",
    URIRef("http://purl.obolibrary.org/obo/RO_0000087"): "has role",
    URIRef("http://purl.obolibrary.org/obo/BFO_0000050"): "is part of",
    URIRef("http://purl.obolibrary.org/obo/CHEBI_50906"): "has functional parent",
    URIRef("http://purl.obolibrary.org/obo/CHEBI_33232"): "has application",
    URIRef("http://purl.obolibrary.org/obo/CHEBI_50906"): "has biological role",
}

# Define High-Value Chemicals (Filter for well-known chemicals)
IMPORTANT_CHEMICALS = {
    "CHEBI:15377",  # Water
    "CHEBI:15422",  # ATP
    "CHEBI:18257",  # Serotonin
    "CHEBI:29309",  # Aspirin
    "CHEBI:22379",  # Penicillin
    "CHEBI:17087",  # Caffeine
    "CHEBI:25512",  # Dopamine
    "CHEBI:57491",  # Glucose
}

# Extract Labels Efficiently
chebi_labels = {}
for s, _, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#label"), None)):
    chebi_id = str(s).split("/")[-1]
    chebi_labels[chebi_id] = str(o)

# Extract Only Relevant Facts (Filter for Common Chemicals)
filtered_facts = set()
for s, p, o in ontology_graph.triples((None, None, None)):
    if p in RELEVANT_RELATIONS:
        # Keep only well-known chemicals OR things directly related to them
        if str(s).split("/")[-1] in IMPORTANT_CHEMICALS or str(o).split("/")[-1] in IMPORTANT_CHEMICALS:
            filtered_facts.add((s, RELEVANT_RELATIONS[p], o))

# Convert to Natural Language
natural_language_statements = list(set([
    f"{chebi_labels.get(str(s).split('/')[-1], str(s))} {relation} {chebi_labels.get(str(o).split('/')[-1], str(o))}."
    for s, relation, o in filtered_facts
]))

# Save Results
with open("chebi_filtered_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print(f"Final filtered ChEBI facts saved to 'chebi_filtered_facts.txt' with {len(natural_language_statements)} meaningful facts.")


Final filtered ChEBI facts saved to 'chebi_filtered_facts.txt' with 0 meaningful facts.


In [7]:
from rdflib import Graph, Namespace, URIRef, Literal
import json

# Step 1: Load ChEBI Ontology (Local)
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")

# Step 2: Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Step 3: Define Relevant Relationships
RELEVANT_RELATIONS = {
    URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"): "is a subclass of",
    URIRef("http://purl.obolibrary.org/obo/RO_0000087"): "has role",
    URIRef("http://purl.obolibrary.org/obo/BFO_0000050"): "is part of",
    URIRef("http://purl.obolibrary.org/obo/CHEBI_50906"): "has functional parent",
    URIRef("http://purl.obolibrary.org/obo/CHEBI_33232"): "has application",
    URIRef("http://purl.obolibrary.org/obo/CHEBI_50906"): "has biological role",
}

# Step 4: Extract Labels for All Entities
chebi_labels = {}
for s, _, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#label"), None)):
    chebi_id = str(s).split("/")[-1]
    chebi_labels[chebi_id] = str(o)

# Step 5: Extract All Inferable Facts
all_facts = []
for s, p, o in ontology_graph.triples((None, None, None)):
    if p in RELEVANT_RELATIONS:
        fact = {
            "subject_id": str(s).split("/")[-1],
            "subject_label": chebi_labels.get(str(s).split("/")[-1], str(s)),
            "predicate": RELEVANT_RELATIONS[p],
            "object_id": str(o).split("/")[-1],
            "object_label": chebi_labels.get(str(o).split("/")[-1], str(o)),
        }
        all_facts.append(fact)

# Step 6: Save All Facts to a JSON File
with open("chebi_inferred_facts.json", "w") as f:
    json.dump(all_facts, f, indent=4)

print(f"Precomputed {len(all_facts)} ChEBI facts saved to 'chebi_inferred_facts.json'.")


Precomputed 374296 ChEBI facts saved to 'chebi_inferred_facts.json'.


In [8]:
import json

# Load the precomputed ChEBI facts
with open("chebi_inferred_facts.json", "r") as f:
    chebi_facts = json.load(f)

# Function to Query Facts by Subject, Predicate, or Object
def query_chebi_facts(subject=None, predicate=None, object_=None):
    results = []
    for fact in chebi_facts:
        if (subject and subject.lower() not in fact["subject_label"].lower()) and subject:
            continue
        if (predicate and predicate.lower() not in fact["predicate"].lower()) and predicate:
            continue
        if (object_ and object_.lower() not in fact["object_label"].lower()) and object_:
            continue
        results.append(fact)
    return results

# Example Queries
print("\n🔎 Query: Facts about 'Aspirin'")
for fact in query_chebi_facts(subject="Aspirin"):
    print(f'{fact["subject_label"]} {fact["predicate"]} {fact["object_label"]}.')

print("\n🔎 Query: Facts where predicate is 'has role'")
for fact in query_chebi_facts(predicate="has role"):
    print(f'{fact["subject_label"]} {fact["predicate"]} {fact["object_label"]}.')



🔎 Query: Facts about 'Aspirin'
aspirin-triggered resolvin D2 is a subclass of secondary allylic alcohol.
aspirin-triggered resolvin D1 is a subclass of N915bdcbe76f3404d937e02c382ac022d.
aspirin-triggered resolvin D4 is a subclass of hydroxy polyunsaturated fatty acid.
aspirin-triggered protectin D1 is a subclass of N9aea547c2ae4438ab11145589be99612.
aspirin-based probe AP is a subclass of salicylates.
aspirin-based probe AP is a subclass of benzoate ester.
aspirin-triggered protectin D1 is a subclass of Nc4b0c1f618064bf8b0b0436a56c35d84.
aspirin-triggered resolvin D4 is a subclass of Nb946d918cf5049de820bb62b760dfade.
aspirin-based probe AP is a subclass of monofluorobenzenes.
aspirin-triggered resolvin D6 is a subclass of Nc334980d07c44f9ba6fb8a25733af094.
aspirin-triggered resolvin D3 is a subclass of triol.
aspirin-based probe AP is a subclass of N2c3151b299284cc8943aedbe9037aa20.
aspirin-triggered resolvin D5 is a subclass of resolvin.
aspirin-triggered resolvin D6 is a subclass 